# Notebook to calculate the template input points for sca25519

In [1]:
import matplotlib.pyplot as plt
import numpy as np 

import sys
sys.path.append('../util/')
import scipy.stats as ss


import numpy as np
from curve25519 import curve25519_base, curve25519

## Template calculation

In [2]:

# In Big Endian
secret_key_a = [0x6A, 0x2C, 0xB9, 0x1D, 0xA5, 0xFB, 0x77, 0xB1,
                      0x2A, 0x99, 0xC0, 0xEB, 0x87, 0x2F, 0x4C, 0xDF,
                      0x45, 0x66, 0xB2, 0x51, 0x72, 0xC1, 0x16, 0x3C,
                      0x7D, 0xA5, 0x18, 0x73, 0x0A, 0x6D, 0x07, 0x70]

# In Little Endian
correct_pk_A = [0x85, 0x20, 0xF0, 0x09, 0x89, 0x30, 0xA7, 0x54,
                            0x74, 0x8B, 0x7D, 0xDC, 0xB4, 0x3E, 0xF7, 0x5A,
                            0x0D, 0xBF, 0x3A, 0x0D, 0x26, 0x38, 0x1A, 0xF4,
                            0xEB, 0xA4, 0xA9, 0x8E, 0xAA, 0x9B, 0x4E, 0x6A]

# In Little Endian
generator_point = [0x09, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00,
                            0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00,
                            0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00,
                            0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00, 0x00]

def flipEndian(arr):
    return arr[::-1]

def packed_to_binary(key):
    bin_key = ""
    for elem in key:
        binary = format(elem, '08b')
        bin_key = bin_key + binary
    return bin_key

def calculate_template_dbladd(secret_key):
    template = 1
    template_progression = []
    template_scalars = []
    binary_key = packed_to_binary(secret_key)
    start_from_msb2 = binary_key.split("1", 1)[1] # keep everything after the first encountered "1" 
    for bit in start_from_msb2:
        double = template * 2
        add = template * 2 + 1
        template_progression.append([double, add])
        template = add if bit == "1" else double
        template_scalars.append(template) 
    return template, template_progression, template_scalars


def calculate_template_montgomory(secret_key):
    ## Without C-swap, so it is not consecutive unlike our target implementation! 
    R0 = 1
    R1 = 2
    template_progression = []
    template_scalars = []
    binary_key = packed_to_binary(secret_key)
    start_from_msb2 = binary_key.split("1", 1)[1] # keep everything after the first encountered "1" 
    for x in start_from_msb2:
        b = 1 - int(x)

        if b == 1:
            R1 = R0 + R1
            R0 = 2 * R0
            double = R0
            add = R1
        elif b == 0:
            R0 = R0 + R1
            R1 = 2 * R1
            double = R1
            add = R0

        template_scalars.append(R0)
        template_progression.append([double, add])

        template = R0
    return template, template_progression, template_scalars


In [3]:
final_scalar, template_progression, template_scalars = calculate_template_montgomory(secret_key_a)

print("Final scalar in int: " + str(final_scalar))
print("Final scalar in hex: 0x" + str(final_scalar.to_bytes(32, byteorder='big').hex())) 
print(template_progression[:6])
print(template_scalars[:6]) 

Final scalar in int: 48024180843069071553745934684982006431825596986621126406018887516696408295280
Final scalar in hex: 0x6a2cb91da5fb77b12a99c0eb872f4cdf4566b25172c1163c7da518730a6d0770
[[4, 3], [6, 7], [14, 13], [26, 27], [54, 53], [106, 107]]
[3, 6, 13, 26, 53, 106]


In [4]:
final_scalar, template_progression, template_scalars = calculate_template_dbladd(secret_key_a)

print("Binary representation of the MSB byte of the key: " + format(secret_key_a[0], '08b'))
print(template_progression[:6])
print(template_scalars[:6]) ## Only recover the first 8 bits => the first 2 bits are fixed by 0, 1


Binary representation of the MSB byte of the key: 01101010
[[2, 3], [6, 7], [12, 13], [26, 27], [52, 53], [106, 107]]
[3, 6, 13, 26, 53, 106]


### Sanity check: 

In [5]:
# Check if scalar matches our secret key
print("Final scalar length binary: " + str(len(bin(final_scalar)[2:])))
print("Final scalar in binary: " + str((bin(final_scalar)[2:])))
print("Final scalar in int: " + str(final_scalar))
print("Final scalar in hex: 0x" + str(final_scalar.to_bytes(32, byteorder='big').hex())) 

Final scalar length binary: 255
Final scalar in binary: 110101000101100101110010001110110100101111110110111011110110001001010101001100111000000111010111000011100101111010011001101111101000101011001101011001001010001011100101100000100010110001111000111110110100101000110000111001100001010011011010000011101110000
Final scalar in int: 48024180843069071553745934684982006431825596986621126406018887516696408295280
Final scalar in hex: 0x6a2cb91da5fb77b12a99c0eb872f4cdf4566b25172c1163c7da518730a6d0770


In [6]:
# Curve implementation from: https://gist.github.com/nickovs/cc3c22d15f239a2640c185035c06f8a3 

# These functions expect all input in little endian
pk_sanity = curve25519_base(flipEndian(secret_key_a)) # Feed secret key in little endian
pk_scalar = curve25519_base(final_scalar.to_bytes(32, byteorder='little'))

pk_sanity_point = curve25519(generator_point, flipEndian(secret_key_a))
pk_scalar_point = curve25519(generator_point, final_scalar.to_bytes(32, byteorder='little'))

print(pk_sanity.hex())
print(pk_scalar.hex())
print(pk_sanity_point.hex())
print(pk_scalar_point.hex())

8520f0098930a754748b7ddcb43ef75a0dbf3a0d26381af4eba4a98eaa9b4e6a
8520f0098930a754748b7ddcb43ef75a0dbf3a0d26381af4eba4a98eaa9b4e6a
8520f0098930a754748b7ddcb43ef75a0dbf3a0d26381af4eba4a98eaa9b4e6a
8520f0098930a754748b7ddcb43ef75a0dbf3a0d26381af4eba4a98eaa9b4e6a


Above public key matches with the one in source code, thus generated points $P$ are correct.

## Sanity check for other scalars
Small piece of code where you can adjust the scalar to verify the calculated point

In [7]:
# In little endian
pk_B = [0xDE, 0x9E, 0xDB, 0x7D, 0x7B, 0x7D, 0xC1, 0xB4,
                            0xD3, 0x5B, 0x61, 0xC2, 0xEC, 0xE4, 0x35, 0x37,
                             0x3F, 0x83, 0x43, 0xC8, 0x5B, 0x78, 0x67, 0x4D,
                             0xAD, 0xFC, 0x7E, 0x14, 0x6F, 0x88, 0x2B, 0x4F]

In [8]:
ss_sanity_check = curve25519(pk_B, flipEndian(secret_key_a))
print(ss_sanity_check.hex())

ss = curve25519(pk_B, final_scalar.to_bytes(32, byteorder="little"))
print(ss.hex())


4a5d9d5ba4ce2de1728e3bf480350f25e07e21c947d19e3376f09b3c1e161742
4a5d9d5ba4ce2de1728e3bf480350f25e07e21c947d19e3376f09b3c1e161742


In [9]:
scalar_hypothesis = 2

point = curve25519(pk_B, scalar_hypothesis.to_bytes(32, byteorder="little"))
print(point.hex())



ab2a7f429c57e360bc4cd2fb11de5252acfac68bf075cd64c4f59009aa604f31


In [10]:
scalar_hypothesis = 2

B2 = curve25519(pk_B, scalar_hypothesis.to_bytes(32, byteorder="little"), return_affine=False) 

print(B2)


(26865443371422354019999921694126971859015032210464741228589280025338407043676, 13626496231957438244516191075570199934064891795316384832256068042236424530943)


## Create collision points on A and A^2

In [11]:
compare_scalars = [2, 3, 6, 7, 12, 13, 26, 27, 52, 53, 106, 107]
scalars = [3, 6, 13, 26, 53, 106]

correct_key = [0,1,1,0,1,0,1,0]

In [12]:
pk_B = [0xDE, 0x9E, 0xDB, 0x7D, 0x7B, 0x7D, 0xC1, 0xB4,
                            0xD3, 0x5B, 0x61, 0xC2, 0xEC, 0xE4, 0x35, 0x37,
                             0x3F, 0x83, 0x43, 0xC8, 0x5B, 0x78, 0x67, 0x4D,
                             0xAD, 0xFC, 0x7E, 0x14, 0x6F, 0x88, 0x2B, 0x4F]

In [15]:
p = 2**255 - 19
A24 = 121666

def fe_add(a,b): return (a + b) % p
def fe_sub(a,b): return (a - b) % p
def fe_mul(a,b): return (a * b) % p
def fe_square(a): return (a * a) % p


class LadderState:
    def __init__(self, xp, zp, xq, zq, x0):
        self.xp = xp % p
        self.zp = zp % p
        self.xq = xq % p
        self.zq = zq % p
        self.x0 = x0 % p
    def copy(self):
        return LadderState(self.xp, self.zp, self.xq, self.zq, self.x0)

def cswap(state: LadderState, condition: int):
    if condition:
        state.xp, state.xq = state.xq, state.xp
        state.zp, state.zq = state.zq, state.zp

def ladderstep(state: LadderState):
    A = fe_add(state.xp, state.zp)
    B = fe_sub(state.xp, state.zp)
    C = fe_add(state.xq, state.zq)
    D = fe_sub(state.xq, state.zq)

    DA = fe_mul(D, A)
    CB = fe_mul(C, B)

    T0 = fe_add(DA, CB)
    T2 = fe_sub(DA, CB)

    X5 = fe_square(T0)
    Z5 = fe_mul(fe_square(T2), state.x0)

    AA = fe_square(A)
    BB = fe_square(B)
    E  = fe_sub(AA, BB)

    X4 = fe_mul(AA, BB)
    Z4 = fe_mul(E, fe_add(BB, fe_mul(E, A24)))

    state.xp, state.zp = X4, Z4
    state.xq, state.zq = X5, Z5
    

In [16]:

def intermediates():
    pk_B_int = int.from_bytes(pk_B, "little")

    A_inters = []
    AA_inters = []

    state = LadderState(xp=1, zp=0, xq=pk_B_int, zq=1, x0=pk_B_int)

    prev_bit = 0
    current_bit = 1
    
    cswap(state, current_bit ^ prev_bit)

    # Inspect state
    # A = fe_add(state.xp, state.zp)
    # AA = fe_square(A)
    # A_inters.append(A)
    # AA_inters.append(AA)

    # Continue ladder
    ladderstep(state)
    prev_bit = current_bit

    # Attackable bits
    for bit in correct_key:

        # Bit 0
        current_bit = 0
        cswap(state, current_bit ^ prev_bit)

        # Inspect state
        A = fe_add(state.xp, state.zp)
        AA = fe_square(A)
        A_inters.append(A)
        AA_inters.append(AA)

        # Bit 1
        current_bit = 1
        cswap(state, current_bit ^ prev_bit)

        # Inspect state
        A = fe_add(state.xp, state.zp)
        AA = fe_square(A)
        A_inters.append(A)
        AA_inters.append(AA)

        # Continue ladder
        ladderstep(state)
        prev_bit = bit # We already know what the correct bit will be

    return A_inters, AA_inters


def generate_point(index, A_inters):
    A = A_inters[index]
    affine_x = fe_sub(A, 1)
    return affine_x


## Sanity check for collision in A and AA

In [17]:
i = 1

A_inters, AA_inters = intermediates()


point = generate_point(i, A_inters)

print("X-affine point: " + str(point))

state = LadderState(xp=1, zp=0, xq=point, zq=1, x0=int.from_bytes(pk_B, "little"))

prev_bit = 0
# Forced MSB = 1
current_bit = 1
cswap(state, current_bit ^ prev_bit)

A = fe_add(state.xp, state.zp)
AA = fe_square(A)

print("Real intermediate A: " , A_inters[i])
print("Calculated intermediate A: " , A)
print("Real intermediate AA: " , AA_inters[i])
print("Calculated intermediate AA: " , AA)



X-affine point: 46933435807174374958983365321381739850621255092601740048069982677240779621398
Real intermediate A:  46933435807174374958983365321381739850621255092601740048069982677240779621399
Calculated intermediate A:  46933435807174374958983365321381739850621255092601740048069982677240779621399
Real intermediate AA:  18922312584091920732306750827810301767070231857038719194512119424828892936115
Calculated intermediate AA:  18922312584091920732306750827810301767070231857038719194512119424828892936115
